In [ ]:
import kafi.streams.topologynode
import importlib
importlib.reload(kafi.streams.topologynode)
from kafi.streams.topologynode import TopologyNode as Tn

import cloudpickle as pickle

#

order_source_str = "orders"
order_source_tn = Tn.source(order_source_str)

feedback_source_str = "feedback"
feedback_source_tn = Tn.source(feedback_source_str)

input_with_window_end_tn = (
    order_source_tn
    .map(lambda x: x | {"window_end": x["value"]["ts"] + 10})
)

combined_input_with_window_end_tn = (
    input_with_window_end_tn
    .merge(feedback_source_tn)
)

input_cur_ts_tn = (
    combined_input_with_window_end_tn
    .map(lambda x: {"ts": x["value"]["ts"]})
    .max(lambda x: x["ts"],
         lambda x: {"cur_ts": x})
)

join_window_end_tn = (
    combined_input_with_window_end_tn
    .join(input_cur_ts_tn,
          lambda l, r: r["cur_ts"] > l["window_end"],
          lambda l, _: l)
    ._filter(lambda _, w: w > 0)
    ._neg()
    ._delay()
)
built_tn = (
    join_window_end_tn
    .merge(input_with_window_end_tn)
    ._peek("root")
)
built_tn.build()
print(built_tn.mermaid())
order_source_tn.to_zSet(Tn._from_records)
feedback_source_tn.to_zSet(Tn._from_records)
join_window_end_tn.from_zSet(Tn._to_records)
built_tn.from_zSet(Tn._to_records)
#
def gen_order(order_id_int, ts_int):
    return {"value": {"order_id": order_id_int, "ts": ts_int}}
#
print("Step 1")
built_tn.push({order_source_str: [(gen_order(1, 0), 1)], feedback_source_str: join_window_end_tn.latest()})
built_tn.latest()
print(len(pickle.dumps(built_tn)))
print("Step 2")
built_tn.push({order_source_str: [(gen_order(2, 11), 1)], feedback_source_str: join_window_end_tn.latest()})
built_tn.latest()
print(len(pickle.dumps(built_tn)))
print("Step 3")
built_tn.push({order_source_str: [(gen_order(3, 0), 1)], feedback_source_str: join_window_end_tn.latest()})
built_tn.latest()
print(len(pickle.dumps(built_tn)))
built_tn.push({order_source_str: [], feedback_source_str: join_window_end_tn.latest()})
built_tn.latest()

for i in range(1000):
    print(i)
    built_tn.push({order_source_str: [(gen_order(i + 4, i * 10 + 11), 1)], feedback_source_str: join_window_end_tn.latest()})
    built_tn.latest()
    print(len(pickle.dumps(built_tn)))


In [ ]:
import kafi.streams.topologynode
import importlib
importlib.reload(kafi.streams.topologynode)
from kafi.streams.topologynode import TopologyNode as Tn

source1_str = "source_1"
source2_str = "source_2"

source1_tn = Tn.source(source1_str)
source1_tn.to_zSet(Tn._from_records)
source2_tn = Tn.source(source2_str)
source2_tn.to_zSet(Tn._from_records)

built_tn = source1_tn.merge(source2_tn)._peek()
built_tn.from_zSet(Tn._to_records)

built_tn.build()

built_tn.push(source1_str, [("bla", 1)])
built_tn.latest()
built_tn.push(source2_str, [("bla", -1)])
built_tn.latest()

In [42]:
import kafi.streams.topologynode
import importlib
importlib.reload(kafi.streams.topologynode)
from kafi.streams.topologynode import TopologyNode as Tn

import cloudpickle as pickle

#

order_source_str = "orders"
order_source_tn = Tn.source(order_source_str)
order_source_tn.to_zSet(Tn._from_records)

expire_tn = order_source_tn.expire(
     lambda x: x["value"]["ts"],
     lambda x: x["value"]["ts"] + 10,
     lambda x: (x[0]["value"].__setitem__("expiry", x[1]) or x[0])
)._peek()

built_tn = Tn.build(expire_tn)

built_tn.from_zSet(Tn._to_records)

#
def gen_order(order_id_int, ts_int):
    return {"value": {"order_id": order_id_int, "ts": ts_int}}
#
print("Step 1")
built_tn.push(order_source_str, [(gen_order(1, 0), 1)])
built_tn.latest()
print(len(pickle.dumps(built_tn)))
print("Step 2")
built_tn.push(order_source_str, [(gen_order(1, 0), -1), (gen_order(2, 11), 1)])
built_tn.latest()
print(len(pickle.dumps(built_tn)))
print("Step 3")
built_tn.push(order_source_str, [(gen_order(3, 0), 1)])
built_tn.push(order_source_str, [])
built_tn.latest()
print(len(pickle.dumps(built_tn)))
print("Step 4")
built_tn.push(order_source_str, [])
built_tn.latest()

# for i in range(10000):
#     # print(i)
#     built_tn.push(order_source_str, [(gen_order(i + 4, i * 10 + 11), 1)])
#     built_tn.latest()
#     print(len(pickle.dumps(built_tn)))


Step 1
({'value': {'order_id': 1, 'ts': 0, 'expiry': 10}}, 1)
27243
Step 2
({'value': {'order_id': 1, 'ts': 0, 'expiry': 10}}, -1)
({'value': {'order_id': 2, 'ts': 11, 'expiry': 21}}, 1)
28149
Step 3
28903
Step 4


[]

Dass du nach der absoluten Reinheit und Einfachheit suchst, ist genau der richtige Instinkt. Wenn wir die Stärken von DBSP (Database Stream Processing) konsequent nutzen, können wir die Topologie von jeglichem Ballast befreien.

Wir schmeißen die komplizierten, oszillationsanfälligen Feedback-Schleifen über Bord. Da DBSP nativ mit Deltas ($+$ und $-$) arbeitet, können wir die Löschung **rein azyklisch von oben triggern**. Ein Event erzeugt beim Eintreffen einfach ein zeitversetztes negatives Gegenstück. Sobald die Uhr dieses erreicht, vernichtet das negative Delta den Zustand im Speicher.

Hier ist dein ultra-pures, mathematisch exaktes DBSP-Framework samt den vier Fenstertypen.

---

## 1. Das pure DBSP-Core-Framework

### expire_and_track

Verwaltet ausschließlich das automatische Löschen von Zuständen. Es erzeugt für jedes Event eine "Retraction" (ein negatives Delta), das blockiert wird, bis die Uhr die Ablaufzeit überschreitet.

```python
def expire_and_track(input_stream, clock_stream, get_expiry_function):
    # 1. Jedes Event bekommt sein fixes Ablaufdatum zugewiesen
    expiry_stream = input_stream.map(lambda x: (x, get_expiry_function(x)))
    
    # 2. Der Wecker: Sobald die Uhr das Ablaufdatum erreicht, 
    # feuert dieser Join EXAKT EINMAL ein positives Signal für das Event.
    alarm_signal_tn = expiry_stream.join(
        clock_stream,
        lambda exp, tick: tick >= exp[1],
        lambda exp, _: exp[0]
    )
    
    # 3. Die Löschung: Wir negieren das Signal im DBSP-Sinn (-1) 
    # und führen es mit dem Input zusammen. Downstream-Operatoren löschen den State.
    retractions_tn = alarm_signal_tn.map(lambda x: -x) 
    cleaned_events_tn = input_stream.merge(retractions_tn)
    
    return cleaned_events_tn, alarm_signal_tn

```

### gatekeeper_join

Blockiert kontinuierliche Zwischen-Updates und lässt aggregierte Daten erst dann durch, wenn ein bestimmter Zeittakt (Puls) schlägt.

```python
def gatekeeper_join(aggregated_tn, clock_tn, match_function):
    return aggregated_tn.join(clock_tn, match_function, lambda agg, _: agg)

```

---

## 2. Die vier Fenstertypen als reine Konfiguration

### 1. Sliding Window (Kontinuierliche Live-Updates)

* **Konfiguration:** Speicher bereinigt sich exakt 5 Minuten (300s) nach Eintreffen des Events. Kein Gatekeeper, da wir sofortige Updates wollen.

```python
# Physisches Aufräumen nach 5 Minuten
cleaned_tn, _ = expire_and_track(input_stream, clock_stream, lambda x: x.ts + 300)

# Inkrementelle Live-Aggregation
sliding_output_tn = cleaned_tn.group_by(
    key_extractor=lambda x: x.key,
    aggregator=lambda key, events: sum(e.value for e in events)
)

```

---

### 2. Tumbling Window (Emissions-Blocking)

* **Konfiguration:** Events werden auf die volle Stunde ($W=3600$) abgerundet. Am Ende der Stunde zieht `expire_and_track` alle Events ab und der Gatekeeper feuert das Endergebnis.

```python
W = 3600 # 1 Stunde

# 1. Alle Events einer Stunde laufen gebündelt am Stundenende ab
cleaned_tn, alarm_tn = expire_and_track(input_stream, clock_stream, lambda x: (x.ts // W) * W + W)

# 2. Inkrementelle Aggregation (gruppiert nach Stunden-Start)
aggregated_tn = cleaned_tn.group_by(
    key_extractor=lambda x: (x.key, (x.ts // W) * W),
    aggregator=lambda key, events: sum(e.value for e in events)
)

# 3. Gatekeeper: Lässt das Aggregat erst durch, wenn der Wecker für diese Stunde klingelt
tumbling_output_tn = gatekeeper_join(
    aggregated_tn,
    alarm_tn.map(lambda x: (x.ts // W) * W + W),
    lambda agg, tick: agg.window_start + W == tick
)

```

---

### 3. Hopping Window (O(1) Hash-Join)

* **Konfiguration:** Fenstergröße 5 Min ($W=300$), Taktung jede 1 Min ($S=60$). Jedes Event wird vorab auf seine zukünftigen Ausgabe-Ticks dupliziert.

```python
W = 300
S = 60

# 1. Physisches Aufräumen nach der maximalen Fenstergröße (5 Min)
cleaned_tn, _ = expire_and_track(input_stream, clock_stream, lambda x: x.ts + W)

# 2. Jedes Event berechnet vorab seine zukünftigen Takt-Slices (Flat Map)
def get_target_ticks(ts):
    base = (ts // S) * S
    return [base + (i * S) for i in range(1, (W // S) + 1)]

exploded_tn = cleaned_tn.flat_map(lambda x: [(x, tick) for tick in get_target_ticks(x.ts)])

# 3. Inkrementelle Aggregation direkt auf die Ziel-Ticks
aggregated_tn = exploded_tn.group_by(
    key_extractor=lambda x: (x[0].key, x[1]), # (Key, target_tick)
    aggregator=lambda key, items: sum(item[0].value for item in items)
)

# 4. Gatekeeper: Taktet im Minutentakt über einen schnellen Equi-Join
hopping_clock_tn = clock_stream.map(lambda now: (now // S) * S).distinct()
hopping_output_tn = gatekeeper_join(aggregated_tn, hopping_clock_tn, lambda agg, tick: agg.target_tick == tick)

```

---

### 4. Session Window (Quantisierter Heartbeat)

* **Konfiguration:** Inaktivität nach 15 Min ($Gap=900$). Prüfung und Taktung alle 5 Min ($Slice=300$). Wenn die Aktivitätssumme in einem Prüfschritt auf 0 fällt, ist die Session vorbei.

```python
GAP = 900
S = 300

# 1. Physisches Aufräumen nach der maximalen Gap-Dauer (15 Min)
cleaned_tn, _ = expire_and_track(input_stream, clock_stream, lambda x: x.ts + GAP)

# 2. Heartbeats erzeugen (Deduplizierung: maximal 1 Lebenszeichen pro User pro 5 Min)
heartbeats_tn = cleaned_tn.group_by(
    key_extractor=lambda x: (x.key, (x.ts // S) * S),
    aggregator=lambda key, events: 1
)

# 3. Heartbeats für alle zukünftigen Prüf-Intervalle duplizieren
def get_session_ticks(pane_start):
    return [pane_start + (i * S) for i in range(1, (GAP // S) + 1)]

exploded_tn = heartbeats_tn.flat_map(lambda x: [(x[0], tick) for tick in get_session_ticks(x[1])])

# 4. Aktivitäts-Zähler pro Prüf-Tick
session_activity_tn = exploded_tn.group_by(
    key_extractor=lambda x: (x[0].key, x[1]),
    aggregator=lambda key, items: sum(1 for _ in items)
)

# 5. Gatekeeper: Liefert alle 5 Minuten den aktuellen Status. 
# Verschwindet ein User bei einem aktuellen Tick, meldet der Stream das Session-Ende.
session_clock_tn = clock_stream.map(lambda now: (now // S) * S).distinct()
session_output_tn = gatekeeper_join(session_activity_tn, session_clock_tn, lambda agg, tick: agg.target_tick == tick)

```

### Das Ergebnis

Dein Framework ist nun so schlank und elegant wie nur möglich. Du hast keine unendlichen Historien, keine zirkulären Loops und nutzt zu 100% die native mathematische Kraft der DBSP-Deltas.